In [14]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [15]:
!pip install -q faiss-cpu
!pip uninstall -y torchao -q

!pip install -q \
transformers==4.56.1 \
peft==0.17.1 \
accelerate==1.10.1 \
sentencepiece \
wandb

In [18]:
!pip install -q faiss-cpu

In [19]:
import re
import string

from glob import glob
from gensim.models import Word2Vec
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

import torch
import faiss

from datasets import load_dataset
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

In [25]:
import os
import wandb

wandb_mode = "online" if os.environ.get("WANDB_API_KEY") else "disabled"

run = wandb.init(
    project="23f2000775-t22026",
    name="milestone3-rag",
    mode=wandb_mode,
    config={
    "embedding_model": embedding_model_name,
    "llm": llm_name,
    "top_k": top_k,
    "sample_size": sample_size
},
    reinit=True
)

wandb.log({
    "MAP@3": rag_map3,
    "Accuracy": rag_accuracy
})

run.finish()

NameError: name 'embedding_model_name' is not defined

In [26]:
train_path = "/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv"

train = pd.read_csv(train_path)

print(train.shape)
train.head()

(2000, 8)


,id,prompt,A,B,C,D,E,answer
0,1,Pick the best possible answer: What is Martin ...,Martin Heidegger believes that humans exist wi...,Martin Heidegger believes that humans do not e...,Martin Heidegger does not believe in the exist...,Martin Heidegger believes that the relationshi...,Martin Heidegger believes that time is an illu...,B
1,2,What is accelerator-based light-ion fusion?,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,A
2,3,Determine the correct option: What is the term...,Blueshifting,Redshifting,Reddening,Whitening,Yellowing,C
3,4,Select the most accurate option: What is Marti...,Martin Heidegger believes that humans exist wi...,Martin Heidegger believes that humans do not e...,Martin Heidegger does not believe in the exist...,Martin Heidegger believes that the relationshi...,Martin Heidegger believes that time is an illu...,B
4,5,Identify the correct statement: What is the co...,"Simultaneity is relative, meaning that two eve...","Simultaneity is relative, meaning that two eve...","Simultaneity is absolute, meaning that two eve...",Simultaneity is a concept that applies only to...,Simultaneity is a concept that applies only to...,A


In [27]:
train.isnull().sum()

id        0
prompt    0
A         0
B         0
C         0
D         0
E         0
answer    0
dtype: int64

In [28]:
text_columns = ['prompt', 'A', 'B', 'C', 'D', 'E']

train[text_columns] = train[text_columns].fillna('')

In [29]:
def clean_text(text):
    text = text.lower()

    for mark in string.punctuation:
        text = text.replace(mark, ' ')

    text = re.sub(r'\s+', ' ', text).strip()

    return text

In [30]:
for column in text_columns:
    train[column + '_clean'] = train[column].apply(clean_text)

train[['prompt', 'prompt_clean']].head()

,prompt,prompt_clean
0,Pick the best possible answer: What is Martin ...,pick the best possible answer what is martin h...
1,What is accelerator-based light-ion fusion?,what is accelerator based light ion fusion
2,Determine the correct option: What is the term...,determine the correct option what is the term ...
3,Select the most accurate option: What is Marti...,select the most accurate option what is martin...
4,Identify the correct statement: What is the co...,identify the correct statement what is the con...


In [31]:
train['prompt_tokens'] = train['prompt_clean'].str.split()

train[['prompt_clean', 'prompt_tokens']].head()

,prompt_clean,prompt_tokens
0,pick the best possible answer what is martin h...,"[pick, the, best, possible, answer, what, is, ..."
1,what is accelerator based light ion fusion,"[what, is, accelerator, based, light, ion, fus..."
2,determine the correct option what is the term ...,"[determine, the, correct, option, what, is, th..."
3,select the most accurate option what is martin...,"[select, the, most, accurate, option, what, is..."
4,identify the correct statement what is the con...,"[identify, the, correct, statement, what, is, ..."


In [32]:
def average_precision_at_3(actual, predicted):
    predicted = predicted[:3]

    if actual not in predicted:
        return 0

    rank = predicted.index(actual) + 1

    return 1 / rank

In [33]:
def map_at_3(actual_answers, predictions):
    scores = []

    for actual, predicted in zip(actual_answers, predictions):
        score = average_precision_at_3(actual, predicted)
        scores.append(score)

    return np.mean(scores)

In [34]:
clean_columns = [
    'prompt_clean',
    'A_clean',
    'B_clean',
    'C_clean',
    'D_clean',
    'E_clean'
]

all_text = []

for column in clean_columns:
    all_text.extend(train[column].tolist())

print('Number of text samples:', len(all_text))

Number of text samples: 12000


In [35]:
tfidf = TfidfVectorizer(
    ngram_range=(1,2),
    stop_words="english",
    max_features=50000,
    sublinear_tf=True
)

tfidf.fit(all_text)

print('TF-IDF vocabulary size:', len(tfidf.vocabulary_))

TF-IDF vocabulary size: 10878


In [36]:
option_labels = ['A', 'B', 'C', 'D', 'E']
tfidf_predictions = []
tfidf_scores = []

for _, row in train.iterrows():
    prompt_vector = tfidf.transform([row['prompt_clean']])

    options = [
        row['A_clean'],
        row['B_clean'],
        row['C_clean'],
        row['D_clean'],
        row['E_clean']
    ]

    option_vectors = tfidf.transform(options)n 

    similarities = cosine_similarity(
        prompt_vector,
        option_vectors
    )[0]

    order = np.argsort(similarities)[::-1]
    prediction = [option_labels[i] for i in order]

    tfidf_predictions.append(prediction)
    tfidf_scores.append(similarities)

In [37]:
train['tfidf_prediction'] = tfidf_predictions

tfidf_map = map_at_3(
    train['answer'].tolist(),
    tfidf_predictions
)

print('TF-IDF MAP@3:', round(tfidf_map, 4))

train[
    ['id', 'answer', 'tfidf_prediction']
].head(10)

TF-IDF MAP@3: 0.285


,id,answer,tfidf_prediction
0,1,B,"[D, C, B, A, E]"
1,2,A,"[C, A, B, D, E]"
2,3,C,"[E, D, C, B, A]"
3,4,B,"[D, C, B, A, E]"
4,5,A,"[E, D, B, C, A]"
5,6,C,"[E, D, C, B, A]"
6,7,E,"[A, C, D, E, B]"
7,8,A,"[D, C, E, B, A]"
8,9,A,"[E, D, C, B, A]"
9,10,A,"[A, C, E, B, D]"


In [38]:
sentences = []

for column in clean_columns:
    for text in train[column]:
        sentences.append(text.split())

print('Number of sentences:', len(sentences))

Number of sentences: 12000


In [39]:
word2vec = Word2Vec(
    sentences=sentences,
    vector_size=100,
    window=5,
    min_count=2,
    workers=4,
    seed=42
)

print('Word2Vec vocabulary size:', len(word2vec.wv))

Word2Vec vocabulary size: 3003


In [40]:
def get_sentence_vector(text):
    vectors = []

    for word in text.split():
        if word in word2vec.wv:
            vectors.append(word2vec.wv[word])

    if len(vectors) == 0:
        return np.zeros(100)

    return np.mean(vectors, axis=0)

In [41]:
word2vec_predictions = []
word2vec_scores = []

for _, row in train.iterrows():
    prompt_vector = get_sentence_vector(
        row['prompt_clean']
    ).reshape(1, -1)

    option_vectors = []

    for label in option_labels:
        option_vector = get_sentence_vector(
            row[label + '_clean']
        )
        option_vectors.append(option_vector)

    option_vectors = np.array(option_vectors)

    similarities = cosine_similarity(
        prompt_vector,
        option_vectors
    )[0]

    order = np.argsort(similarities)[::-1]
    prediction = [option_labels[i] for i in order]

    word2vec_predictions.append(prediction)
    word2vec_scores.append(similarities)

In [42]:
train['word2vec_prediction'] = word2vec_predictions

word2vec_map = map_at_3(
    train['answer'].tolist(),
    word2vec_predictions
)

print('Word2Vec MAP@3:', round(word2vec_map, 4))

Word2Vec MAP@3: 0.328


In [43]:
#final

In [44]:
# %% [markdown]
# ## Leaderboard model â€” DeBERTa-v3 + LoRA multiple-choice classifier
#
# Paste these cells *after* the existing notebook cells.  This deliberately
# leaves the TF-IDF/Word2Vec work intact and does not use generation or RAG.

# %%
# Install only the packages required by the transformer block.  Restart the
# notebook kernel once if Kaggle upgrades a package during this cell.
!pip install -q -U "transformers>=4.45,<5" "peft>=0.13" "accelerate>=0.34" "sentencepiece>=0.2" "wandb>=0.18"

# %%
import gc
import glob
import os
import random
from dataclasses import dataclass

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
import wandb
from datasets import Dataset
from sklearn.metrics import accuracy_score, f1_score
from sklearn.model_selection import train_test_split
from transformers import (
    AutoModelForMultipleChoice,
    AutoTokenizer,
    EarlyStoppingCallback,
    Trainer,
    TrainingArguments,
    set_seed,
)

# Kaggle settings: switch to deberta-v3-large only after confirming it fits.
# v3-base is the default because it allows a larger effective batch on a T4.
CFG = {
    "model_name": "microsoft/deberta-v3-base",
    "max_length": 384,
    "learning_rate": 1.5e-5,
    "train_batch_size": 4,
    "eval_batch_size": 8,
    "gradient_accumulation": 4,
    "epochs": 6,
    "weight_decay": 0.01,
    "warmup_ratio": 0.08,
    "label_smoothing": 0.0,
    "validation_size": 0.12,
    "seed": 42,
    "retrain_on_full_train": True,
}

# A compact, deliberately conservative search grid.  For a deadline run,
# retain the defaults above; use this grid for a second notebook version only.
# SEARCH_SPACE = {
#     "learning_rate": [1e-5, 1.5e-5, 2e-5],
#     "train_batch_size": [4, 8],
#     "epochs": [3, 4],
#     "lora_r": [8, 16, 32],
#     "weight_decay": [0.0, 0.01],
#     "warmup_ratio": [0.05, 0.08, 0.10],
# }

set_seed(CFG["seed"])
torch.backends.cuda.matmul.allow_tf32 = False
assert torch.cuda.is_available(), "Enable a Kaggle GPU accelerator before training."
print(torch.cuda.get_device_name(0))
print("GPU count:", torch.cuda.device_count())

# %%
# Locate competition files without assuming whether Kaggle mounted them under
# /competitions or directly under /kaggle/input/<competition>.
def locate_competition_file(filename):
    candidates = glob.glob(f"/kaggle/input/**/{filename}", recursive=True)
    if not candidates:
        raise FileNotFoundError(f"Could not find {filename} below /kaggle/input")
    preferred = [p for p in candidates if "smart-mcq-solver-challenge" in p]
    return preferred[0] if preferred else candidates[0]

train_path = locate_competition_file("train.csv")
test_path = locate_competition_file("test.csv")
sample_paths = glob.glob("/kaggle/input/**/sample_submission.csv", recursive=True)

# Re-read raw text: transformer tokenization should preserve punctuation and
# casing, unlike the previous baseline's cleaned columns.
mcq_train = pd.read_csv(train_path)
mcq_test = pd.read_csv(test_path)
OPTION_LABELS = [c for c in ["A", "B", "C", "D", "E"] if c in mcq_train.columns]
assert len(OPTION_LABELS) >= 2, f"Expected option columns, got {mcq_train.columns.tolist()}"
assert "answer" in mcq_train.columns, "The training CSV must include answer."
assert set(mcq_train.answer.unique()).issubset(set(OPTION_LABELS))

for frame in (mcq_train, mcq_test):
    for col in ["prompt", *OPTION_LABELS]:
        frame[col] = frame[col].fillna("").astype(str)

label_to_id = {label: i for i, label in enumerate(OPTION_LABELS)}
mcq_train["labels"] = mcq_train["answer"].map(label_to_id).astype("int64")
print("Train/test:", mcq_train.shape, mcq_test.shape, "choices:", OPTION_LABELS)
display(mcq_train[["prompt", *OPTION_LABELS, "answer"]].head(2))

# %%
tokenizer = AutoTokenizer.from_pretrained(
    CFG["model_name"],
    use_fast=False,
)

def tokenize_mcq(batch, with_labels=True):
    num_rows = len(batch["prompt"])

    first_sentences = [[q] * len(OPTION_LABELS) for q in batch["prompt"]]
    second_sentences = [
        [f"{label}. {batch[label][i]}" for label in OPTION_LABELS]
        for i in range(num_rows)
    ]

    first_sentences = sum(first_sentences, [])
    second_sentences = sum(second_sentences, [])

    tokenized = tokenizer(
        first_sentences,
        second_sentences,
        truncation=True,
        max_length=CFG["max_length"],
    )

    tokenized = {
        k: [
            v[i:i + len(OPTION_LABELS)]
            for i in range(0, len(v), len(OPTION_LABELS))
        ]
        for k, v in tokenized.items()
    }

    if with_labels:
        tokenized["labels"] = batch["labels"]

    return tokenized

@dataclass
class MultipleChoiceCollator:
    tokenizer: object

    def __call__(self, features):
        has_labels = "labels" in features[0]
        labels = [feature["labels"] for feature in features] if has_labels else None
        num_choices = len(features[0]["input_ids"])

        flattened = [
            {
                key: value[choice_index]
                for key, value in feature.items()
                if key != "labels"
            }
            for feature in features
            for choice_index in range(num_choices)
        ]

        batch = self.tokenizer.pad(
            flattened,
            padding=True,
            return_tensors="pt",
        )
        batch = {
            key: value.view(len(features), num_choices, -1)
            for key, value in batch.items()
        }

        if has_labels:
            batch["labels"] = torch.tensor(labels, dtype=torch.long)

        return batch

collator = MultipleChoiceCollator(tokenizer)

def make_dataset(frame, include_labels):
    columns = ["prompt", *OPTION_LABELS] + (["labels"] if include_labels else [])
    ds = Dataset.from_pandas(frame[columns], preserve_index=False)
    return ds.map(
        lambda batch: tokenize_mcq(batch, with_labels=include_labels),
        batched=True,
        remove_columns=ds.column_names,
        desc="Tokenizing MCQs",
    )

train_indices, valid_indices = train_test_split(
    np.arange(len(mcq_train)),
    test_size=CFG["validation_size"],
    random_state=CFG["seed"],
    stratify=mcq_train["labels"],
)
train_ds = make_dataset(mcq_train.iloc[train_indices].reset_index(drop=True), include_labels=True)
valid_ds = make_dataset(mcq_train.iloc[valid_indices].reset_index(drop=True), include_labels=True)
test_ds = make_dataset(mcq_test.reset_index(drop=True), include_labels=False)
print(f"Split: {len(train_ds):,} train / {len(valid_ds):,} validation")

# %%
def make_model():
    model = AutoModelForMultipleChoice.from_pretrained(
        CFG["model_name"]
    )

    model.config.use_cache = False
    #model.gradient_checkpointing_enable()

    total_params = sum(p.numel() for p in model.parameters())
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

    print(f"Trainable params: {trainable_params:,}")
    print(f"Total params: {total_params:,}")

    return model

def map3_from_logits(logits, labels):
    ranks = np.argsort(-logits, axis=1)
    reciprocal_rank = np.array([
        1.0 / (np.where(rank == truth)[0][0] + 1)
        if truth in rank[:3] else 0.0
        for rank, truth in zip(ranks, labels)
    ])
    return float(reciprocal_rank.mean())

def compute_metrics(eval_prediction):
    logits, labels = eval_prediction
    if isinstance(logits, tuple):
        logits = logits[0]
    prediction_ids = np.argmax(logits, axis=1)
    return {
        "accuracy": accuracy_score(labels, prediction_ids),
        "macro_f1": f1_score(labels, prediction_ids, average="macro", zero_division=0),
        "map_at_3": map3_from_logits(logits, labels),
    }

# %%
# Do not set WANDB_MODE=disabled: the required Kaggle run is created here.
# In Kaggle, add WANDB_API_KEY as a Notebook secret before running this cell.
from kaggle_secrets import UserSecretsClient
import wandb

user_secrets = UserSecretsClient()
wandb_key = user_secrets.get_secret("WANDB_API_KEY")
if wandb.run is not None:
    wandb.finish()
wandb.login(key=wandb_key)

wandb_run = wandb.init(
    project="23f2000775-t22026",
    name=f"deberta-v3-base-full-ft",
    config={**CFG},
    reinit=True,
)

wandb.define_metric("train/global_step")
wandb.define_metric("train/*", step_metric="train/global_step")
wandb.define_metric("eval/*", step_metric="train/global_step")

training_args = TrainingArguments(
    output_dir="/kaggle/working/deberta_mcq",
    learning_rate=CFG["learning_rate"],
    per_device_train_batch_size=CFG["train_batch_size"],
    per_device_eval_batch_size=CFG["eval_batch_size"],
    gradient_accumulation_steps=CFG["gradient_accumulation"],
    num_train_epochs=CFG["epochs"],
    weight_decay=CFG["weight_decay"],
    warmup_ratio=CFG["warmup_ratio"],
    lr_scheduler_type="cosine",
    fp16=True,
    tf32=False,
    gradient_checkpointing=False,
    label_smoothing_factor=CFG["label_smoothing"],
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="steps",
    logging_steps=20,
    save_total_limit=1,
    load_best_model_at_end=True,
    metric_for_best_model="eval_map_at_3",
    greater_is_better=True,
    report_to=["wandb"],
    run_name=wandb_run.name,
    seed=CFG["seed"],
    dataloader_num_workers=2,
    optim="adamw_torch",
)


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 1.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 87.3 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 775.8/775.8 kB 42.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 389.2/389.2 kB 26.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 39.1 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 26.5/26.5 MB 67.4 MB/s eta 0:00:00:00:0100:01
Tesla T4
GPU count: 2
Train/test: (2000, 9) (500, 7) choices: ['A', 'B', 'C', 'D', 'E']


,prompt,A,B,C,D,E,answer
0,Pick the best possible answer: What is Martin ...,Martin Heidegger believes that humans exist wi...,Martin Heidegger believes that humans do not e...,Martin Heidegger does not believe in the exist...,Martin Heidegger believes that the relationshi...,Martin Heidegger believes that time is an illu...,B
1,What is accelerator-based light-ion fusion?,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,A


tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/579 [00:00<?, ?B/s]

spm.model:   0%|          | 0.00/2.46M [00:00<?, ?B/s]

Tokenizing MCQs:   0%|          | 0/1760 [00:00<?, ? examples/s]

Tokenizing MCQs:   0%|          | 0/240 [00:00<?, ? examples/s]

Tokenizing MCQs:   0%|          | 0/500 [00:00<?, ? examples/s]

Split: 1,760 train / 240 validation


/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: ramakrishna12-2005 (ramakrishna12-2005-indian-institute-of-technology-madras) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
wandb: WARNING Using a boolean value for 'reinit' is deprecated. Use 'return_previous' or 'finish_previous' instead.


In [45]:

model = make_model()
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=valid_ds,
    data_collator=collator,
    processing_class=tokenizer,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2, early_stopping_threshold=0.001)],
)
trainer.train()
validation_metrics = trainer.evaluate(valid_ds, metric_key_prefix="validation")
print({k: round(v, 5) for k, v in validation_metrics.items() if isinstance(v, float)})
wandb.log({
    "validation/MAP@3": validation_metrics["validation_map_at_3"],
    "validation/accuracy": validation_metrics["validation_accuracy"],
    "validation/macro_f1": validation_metrics["validation_macro_f1"],
})
wandb.log({"validation/best_checkpoint": trainer.state.best_model_checkpoint or "final"})


if CFG["retrain_on_full_train"]:
    best_epoch = max(1, int(round(float(trainer.state.best_metric and trainer.state.epoch or CFG["epochs"]))))
    del trainer, model
    gc.collect()
    torch.cuda.empty_cache()

    full_train_ds = make_dataset(mcq_train.reset_index(drop=True), include_labels=True)
    final_args = TrainingArguments(
        output_dir="/kaggle/working/deberta_mcq_final",
        learning_rate=CFG["learning_rate"],
        per_device_train_batch_size=CFG["train_batch_size"],
        gradient_accumulation_steps=CFG["gradient_accumulation"],
        num_train_epochs=best_epoch,
        weight_decay=CFG["weight_decay"],
        warmup_ratio=CFG["warmup_ratio"],
        lr_scheduler_type="cosine",
        fp16=True, tf32=False, gradient_checkpointing=False,
        label_smoothing_factor=CFG["label_smoothing"],
        logging_strategy="steps", logging_steps=20,
        save_strategy="no", report_to=["wandb"], run_name=wandb_run.name,
        seed=CFG["seed"], dataloader_num_workers=2, optim="adamw_torch_fused",
    )
    model = make_model()
    trainer = Trainer(
        model=model, args=final_args, train_dataset=full_train_ds,
        data_collator=collator, processing_class=tokenizer,
    )
    trainer.train()

# %%


pytorch_model.bin:   0%|          | 0.00/371M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/371M [00:00<?, ?B/s]

Some weights of DebertaV2ForMultipleChoice were not initialized from the model checkpoint at microsoft/deberta-v3-base and are newly initialized: ['classifier.bias', 'classifier.weight', 'pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Trainable params: 184,422,913
Total params: 184,422,913


The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 2, 'bos_token_id': 1}.
/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


...

Tokenizing MCQs:   0%|          | 0/2000 [00:00<?, ? examples/s]

Some weights of DebertaV2ForMultipleChoice were not initialized from the model checkpoint at microsoft/deberta-v3-base and are newly initialized: ['classifier.bias', 'classifier.weight', 'pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Trainable params: 184,422,913
Total params: 184,422,913


The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 2, 'bos_token_id': 1}.
/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Step,Training Loss
20,1.611800
40,1.602200
60,1.515200
80,1.236100
100,1.104600
120,0.864800
140,0.755900
160,0.656700
180,0.571100
200,0.452400


In [46]:


# Predict on test set
prediction_output = trainer.predict(test_ds)
logits = prediction_output.predictions

# Top-3 predictions
top3_idx = np.argsort(logits, axis=1)[:, -3:][:, ::-1]
top3_predictions = [
    " ".join(OPTION_LABELS[i] for i in row)
    for row in top3_idx
]

# Build submission
if sample_paths:
    submission = pd.read_csv(sample_paths[0])

    # Automatically detect the prediction column
    pred_col = [c for c in submission.columns if c.lower() not in ["id"]][0]

    submission[pred_col] = top3_predictions
else:
    submission = pd.DataFrame({
        "ID": mcq_test["ID"] if "ID" in mcq_test.columns else mcq_test["id"],
        "Prediction": top3_predictions
    })

submission.to_csv("/kaggle/working/submission.csv", index=False)

print(submission.head())
print("\nSaved to /kaggle/working/submission.csv")

   ID Prediction
0   1      A D C
1   2      B A C
2   3      B E D
3   4      E C A
4   5      C D A

Saved to /kaggle/working/submission.csv


In [47]:
logits = trainer.predict(test_ds).predictions
if isinstance(logits, tuple):
    logits = logits[0]

probs_seed42 = torch.softmax(torch.tensor(logits), dim=1).cpu().numpy()
np.save("/kaggle/working/probs_seed42.npy", probs_seed42)

In [ ]:
#######

In [48]:
def train_full_seed(seed, epochs=5):
    set_seed(seed)
    np.random.seed(seed)
    random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    full_train_ds = make_dataset(
        mcq_train.reset_index(drop=True),
        include_labels=True,
    )

    args = TrainingArguments(
        output_dir=f"/kaggle/working/deberta_seed_{seed}",
        learning_rate=1.5e-5,
        per_device_train_batch_size=4,
        per_device_eval_batch_size=8,
        gradient_accumulation_steps=4,
        num_train_epochs=epochs,
        weight_decay=0.01,
        warmup_ratio=0.08,
        lr_scheduler_type="cosine",
        fp16=True,
        gradient_checkpointing=False,
        logging_strategy="steps",
        logging_steps=20,
        save_strategy="no",
        report_to="none",
        seed=seed,
        dataloader_num_workers=2,
        optim="adamw_torch",
    )

    model = make_model()
    seed_trainer = Trainer(
        model=model,
        args=args,
        train_dataset=full_train_ds,
        data_collator=collator,
        processing_class=tokenizer,
    )
    seed_trainer.train()

    logits = seed_trainer.predict(test_ds).predictions
    if isinstance(logits, tuple):
        logits = logits[0]

    probabilities = torch.softmax(torch.tensor(logits), dim=1).cpu().numpy()
    np.save(f"/kaggle/working/probs_seed{seed}.npy", probabilities)

    del seed_trainer, model
    gc.collect()
    torch.cuda.empty_cache()

train_full_seed(seed=2028, epochs=5)

Tokenizing MCQs:   0%|          | 0/2000 [00:00<?, ? examples/s]

Some weights of DebertaV2ForMultipleChoice were not initialized from the model checkpoint at microsoft/deberta-v3-base and are newly initialized: ['classifier.bias', 'classifier.weight', 'pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Trainable params: 184,422,913
Total params: 184,422,913


The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 2, 'bos_token_id': 1}.
/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Step,Training Loss
20,1.607600
40,1.601000
60,1.495200
80,1.166200
100,0.846200
120,0.555100
140,0.366200
160,0.257700
180,0.181600
200,0.141300


In [51]:
train_full_seed(seed=2027, epochs=5)

Tokenizing MCQs:   0%|          | 0/2000 [00:00<?, ? examples/s]

Some weights of DebertaV2ForMultipleChoice were not initialized from the model checkpoint at microsoft/deberta-v3-base and are newly initialized: ['classifier.bias', 'classifier.weight', 'pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Trainable params: 184,422,913
Total params: 184,422,913


The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 2, 'bos_token_id': 1}.
/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Step,Training Loss
20,1.611400
40,1.602200
60,1.447600
80,0.978300
100,0.656100
120,0.434100
140,0.262800
160,0.178100
180,0.138100
200,0.111400


In [54]:
import os

for seed in [42, 2027, 2028]:
    print(seed, os.path.exists(f"/kaggle/working/probs_seed{seed}.npy"))

42 True
2027 True
2028 True


In [53]:
p42 = np.load("/kaggle/working/probs_seed42.npy")
p27 = np.load("/kaggle/working/probs_seed2027.npy")
p28 = np.load("/kaggle/working/probs_seed2028.npy")

ensemble_probs = (p42 + p27 + p28) / 3
top3_indices = np.argsort(-ensemble_probs, axis=1)[:, :3]

submission = pd.read_csv(sample_paths[0])
submission["Prediction"] = [
    " ".join(OPTION_LABELS[i] for i in row)
    for row in top3_indices
]

submission.to_csv("/kaggle/working/submission_ensemble.csv", index=False)
display(submission.head())

,ID,Prediction
0,1,A D B
1,2,B A C
2,3,B E D
3,4,E C A
4,5,C D A


In [ ]:
# DeBERTa-v3-large diversity model
LARGE_CFG={"model_name":"microsoft/deberta-v3-large","max_length":320,"learning_rate":1e-5,"train_batch_size":1,"gradient_accumulation":8,"epochs":3,"weight_decay":0.01,"warmup_ratio":0.06,"seed":2029}
set_seed(LARGE_CFG["seed"])
large_tokenizer=AutoTokenizer.from_pretrained(LARGE_CFG["model_name"],use_fast=False)
def tokenize_large(batch):
    first,second=[],[]
    for i,question in enumerate(batch["prompt"]):
        first.extend([f"Question: {question}"]*len(OPTION_LABELS))
        second.extend([f"Answer: {batch[label][i]}" for label in OPTION_LABELS])
    encoded=large_tokenizer(first,second,truncation=True,max_length=LARGE_CFG["max_length"],padding=False)
    return {key:[value[i:i+len(OPTION_LABELS)] for i in range(0,len(value),len(OPTION_LABELS))] for key,value in encoded.items()}
large_train_ds=Dataset.from_pandas(mcq_train[["prompt",*OPTION_LABELS,"labels"]].reset_index(drop=True),preserve_index=False)
large_train_ds=large_train_ds.map(tokenize_large,batched=True,remove_columns=["prompt",*OPTION_LABELS,"labels"],desc="Tokenizing large train")
large_collator=MultipleChoiceCollator(large_tokenizer)
large_args=TrainingArguments(output_dir="/kaggle/working/deberta_large_final",learning_rate=1e-5,per_device_train_batch_size=1,gradient_accumulation_steps=8,num_train_epochs=3,weight_decay=0.01,warmup_ratio=0.06,lr_scheduler_type="cosine",fp16=True,gradient_checkpointing=False,logging_strategy="steps",logging_steps=20,save_strategy="no",report_to="none",seed=2029,dataloader_num_workers=2,optim="adamw_torch")
large_model=AutoModelForMultipleChoice.from_pretrained(LARGE_CFG["model_name"])
large_trainer=Trainer(model=large_model,args=large_args,train_dataset=large_train_ds,data_collator=large_collator,processing_class=large_tokenizer)
large_trainer.train()
large_test_ds=Dataset.from_pandas(mcq_test[["prompt",*OPTION_LABELS]].reset_index(drop=True),preserve_index=False)
large_test_ds=large_test_ds.map(tokenize_large,batched=True,remove_columns=["prompt",*OPTION_LABELS],desc="Tokenizing large test")
large_output=large_trainer.predict(large_test_ds).predictions
if isinstance(large_output,tuple): large_output=large_output[0]
large_probs=torch.softmax(torch.tensor(large_output),dim=1).cpu().numpy()
np.save("/kaggle/working/probs_large.npy",large_probs)
print("Saved large-model probabilities:",large_probs.shape)


In [ ]:
base_probs=(np.load("/kaggle/working/probs_seed42.npy")+np.load("/kaggle/working/probs_seed2027.npy")+np.load("/kaggle/working/probs_seed2028.npy"))/3
large_probs=np.load("/kaggle/working/probs_large.npy")
for large_weight in (0.15,0.25,0.35):
    blended_probs=(1.0-large_weight)*base_probs+large_weight*large_probs
    blended_indices=np.argsort(-blended_probs,axis=1)[:,:3]
    candidate=pd.read_csv(sample_paths[0])
    candidate["Prediction"]=[" ".join(OPTION_LABELS[i] for i in row) for row in blended_indices]
    path=f"/kaggle/working/submission_large_{int(large_weight*100)}.csv"
    candidate.to_csv(path,index=False)
    print("Saved",path)
submission=pd.read_csv("/kaggle/working/submission_large_25.csv")
submission.to_csv("/kaggle/working/submission_improved.csv",index=False)
display(submission.head())
